# TD4

# Step 1 

We prepared a file named private_kb_aligned_ttl

# Step 2 : Entity Linking with Open Knowledge Bases

In [5]:
import re
import requests
import pandas as pd
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL

# =========================
# CONFIG
# =========================
INPUT_FILE = r"C:\ESILV\Web\TD4\private_kb_v2.ttl"
OUTPUT_FILE = "private_kb_aligned.ttl"
MAPPING_FILE = "mapping.csv"
NEW_ENTITIES_FILE = "new_entities_ontology.ttl"

WIKIDATA_API = "https://www.wikidata.org/w/api.php"

HEADERS = {
    "User-Agent": "ESILV-KB-Linking/1.0 (student project)"
}

# =========================
# NAMESPACES
# =========================
EX = Namespace("http://example.org/")
WD = Namespace("http://www.wikidata.org/entity/")

# =========================
# LOAD GRAPH
# =========================
g = Graph()
g.parse(INPUT_FILE, format="turtle")

new_entities_graph = Graph()
new_entities_graph.bind("ex", EX)
new_entities_graph.bind("rdf", RDF)
new_entities_graph.bind("rdfs", RDFS)
new_entities_graph.bind("owl", OWL)

# =========================
# HELPERS
# =========================
def uri_local_name(uri):
    uri = str(uri)
    if "#" in uri:
        return uri.split("#")[-1]
    return uri.rstrip("/").split("/")[-1]

def camel_to_label(name):
    name = re.sub(r"([a-z])([A-Z])", r"\1 \2", name)
    name = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1 \2", name)
    return name.strip()

def get_entity_label(graph, entity):
    for _, _, label in graph.triples((entity, RDFS.label, None)):
        if isinstance(label, Literal):
            return str(label)
    return camel_to_label(uri_local_name(entity))

def search_wikidata(entity_name, entity_type=None):
    params = {
        "action": "wbsearchentities",
        "search": entity_name,
        "language": "en",
        "format": "json",
        "limit": 10
    }

    try:
        response = requests.get(
            WIKIDATA_API,
            params=params,
            headers=HEADERS,
            timeout=15
        )
        response.raise_for_status()
        data = response.json()

        results = data.get("search", [])
        print(f"DEBUG search='{entity_name}' -> {len(results)} results")

        if not results:
            return None, 0.0, None

        # mots-clés attendus selon le type RDF privé
        expected_keywords = []
        if entity_type == EX.Film:
            expected_keywords = ["film", "movie"]
        elif entity_type == EX.Person:
            expected_keywords = [
                "actor", "actress", "director", "filmmaker",
                "producer", "screenwriter", "person"
            ]
        elif entity_type == EX.Actor:
            expected_keywords = ["actor", "actress"]
        elif entity_type == EX.Director:
            expected_keywords = ["director", "filmmaker"]
        elif entity_type == EX.Genre:
            expected_keywords = ["genre", "film genre"]
        elif entity_type == EX.Award:
            expected_keywords = ["award", "prize"]
        elif entity_type == EX.Studio:
            expected_keywords = ["studio", "company", "film studio"]
        elif entity_type == EX.Country:
            expected_keywords = ["country", "state"]
        elif entity_type == EX.City:
            expected_keywords = ["city"]

        scored_candidates = []

        for result in results:
            wd_id = result.get("id")
            label = result.get("label", "")
            description = result.get("description", "")

            score = 0

            search_lower = entity_name.lower().strip()
            label_lower = label.lower().strip()
            desc_lower = description.lower().strip()

            # score sur le label
            if label_lower == search_lower:
                score += 100
            elif search_lower in label_lower or label_lower in search_lower:
                score += 60

            # bonus si la description correspond au type attendu
            for kw in expected_keywords:
                if kw in desc_lower:
                    score += 40

            scored_candidates.append((score, wd_id, label, description))

        scored_candidates.sort(reverse=True, key=lambda x: x[0])
        best_score, best_id, best_label, best_description = scored_candidates[0]

        if best_score >= 120:
            confidence = 0.99
        elif best_score >= 80:
            confidence = 0.90
        elif best_score >= 50:
            confidence = 0.75
        else:
            confidence = 0.50

        return best_id, confidence, best_description

    except Exception as e:
        print(f"ERROR for '{entity_name}': {e}")
        return None, 0.0, None

def is_private_uri(node):
    return isinstance(node, URIRef) and str(node).startswith(str(EX))

def is_class_entity(graph, entity):
    return (entity, RDF.type, RDFS.Class) in graph or (entity, RDF.type, OWL.Class) in graph

def is_property_entity(graph, entity):
    return (entity, RDF.type, RDF.Property) in graph or (entity, RDF.type, OWL.ObjectProperty) in graph

def get_entity_types(graph, entity):
    return list(graph.objects(entity, RDF.type))

def should_link_entity(graph, entity):
    if is_class_entity(graph, entity):
        return False
    if is_property_entity(graph, entity):
        return False

    entity_types = set(get_entity_types(graph, entity))

    allowed_types = {
        EX.Film,
        EX.Person,
        EX.Actor,
        EX.Director,
        EX.Composer,
        EX.Studio,
        EX.Country,
        EX.Language,
        EX.Genre,
        EX.Franchise,
        EX.Award,
        EX.AwardCategory,
        EX.Character,
        EX.City
    }

    return len(entity_types.intersection(allowed_types)) > 0

def add_new_entity_semantics(graph_out, entity, original_graph):
    for o in original_graph.objects(entity, RDF.type):
        graph_out.add((entity, RDF.type, o))

    found_label = False
    for o in original_graph.objects(entity, RDFS.label):
        graph_out.add((entity, RDFS.label, o))
        found_label = True

    if not found_label:
        graph_out.add((entity, RDFS.label, Literal(get_entity_label(original_graph, entity), lang="en")))

# =========================
# EXTRACT ENTITIES
# =========================
all_private_nodes = set()

for s, p, o in g:
    if is_private_uri(s):
        all_private_nodes.add(s)
    if is_private_uri(o):
        all_private_nodes.add(o)

entities_to_link = sorted(
    [e for e in all_private_nodes if should_link_entity(g, e)],
    key=lambda x: uri_local_name(x)
)

print(f"Found {len(all_private_nodes)} private URIs")
print(f"Entities selected for linking: {len(entities_to_link)}")

print("\nSelected entities:")
for e in entities_to_link:
    print("-", get_entity_label(g, e), "|", list(g.objects(e, RDF.type)))

# =========================
# LINKING
# =========================
mapping_data = []

for entity in entities_to_link:
    search_name = get_entity_label(g, entity)
    wd_id, confidence, description = search_wikidata(search_name)

    if wd_id:
        wd_uri = WD[wd_id]
        g.add((entity, OWL.sameAs, wd_uri))
        mapping_data.append([
            str(entity),
            str(wd_uri),
            round(confidence, 2),
            search_name,
            description if description else ""
        ])
        print(f"Linked: {search_name} -> {wd_id} (confidence={confidence:.2f})")
    else:
        add_new_entity_semantics(new_entities_graph, entity, g)
        mapping_data.append([
            str(entity),
            "NEW_ENTITY",
            0.0,
            search_name,
            "Not found in Wikidata"
        ])
        print(f"Not found: {search_name}")

# =========================
# SAVE FILES
# =========================
g.serialize(OUTPUT_FILE, format="turtle")
new_entities_graph.serialize(NEW_ENTITIES_FILE, format="turtle")

df = pd.DataFrame(
    mapping_data,
    columns=["Private Entity", "External URI", "Confidence", "Search Label", "Description"]
)
df.to_csv(MAPPING_FILE, index=False)

print("\nDone.")
print(f"Aligned KB saved as: {OUTPUT_FILE}")
print(f"Mapping table saved as: {MAPPING_FILE}")
print(f"New entities ontology saved as: {NEW_ENTITIES_FILE}")

Found 46 private URIs
Entities selected for linking: 8

Selected entities:
- Christopher Nolan | [rdflib.term.URIRef('http://example.org/Person')]
- Forrest Gump | [rdflib.term.URIRef('http://example.org/Film')]
- Inception | [rdflib.term.URIRef('http://example.org/Film')]
- Interstellar | [rdflib.term.URIRef('http://example.org/Film')]
- Leonardo Di Caprio | [rdflib.term.URIRef('http://example.org/Person')]
- The Dark Knight | [rdflib.term.URIRef('http://example.org/Film')]
- Titanic | [rdflib.term.URIRef('http://example.org/Film')]
- Tom Hanks | [rdflib.term.URIRef('http://example.org/Person')]
DEBUG search='Christopher Nolan' -> 9 results
Linked: Christopher Nolan -> Q25191 (confidence=0.90)
DEBUG search='Forrest Gump' -> 10 results
Linked: Forrest Gump -> Q552213 (confidence=0.90)
DEBUG search='Inception' -> 10 results
Linked: Inception -> Q105824503 (confidence=0.90)
DEBUG search='Interstellar' -> 10 results
Linked: Interstellar -> Q13417189 (confidence=0.90)
DEBUG search='Leonard

# Step 3 : Predicate Alignment Using SPARQL

In [6]:
!pip install SPARQLWrfrom SPARQLWrapper import SPARQLWrapper, JSON


SyntaxError: invalid syntax (1178648359.py, line 1)

In [8]:
from SPARQLWrapper import SPARQLWrapper, JSON
from rdflib import Graph, Namespace, URIRef
from rdflib.namespace import RDF, RDFS, OWL
import pandas as pd

# =========================
# CONFIG
# =========================
INPUT_FILE = r"C:\ESILV\Web\TD4\private_kb_aligned.ttl"
OUTPUT_FILE = "predicate_alignment.csv"

EX = Namespace("http://example.org/")
WD = "http://www.wikidata.org/entity/"
WDT = "http://www.wikidata.org/prop/direct/"

# =========================
# LOAD GRAPH
# =========================
g = Graph()
g.parse(INPUT_FILE, format="turtle")

# =========================
# SPARQL ENDPOINT
# =========================
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

# Important pour Wikidata
sparql.addCustomHttpHeader(
    "User-Agent",
    "ESILV-KB-Predicate-Alignment/1.0 (student project)"
)

# =========================
# HELPERS
# =========================
def uri_local_name(uri):
    uri = str(uri)
    if "#" in uri:
        return uri.split("#")[-1]
    return uri.rstrip("/").split("/")[-1]

def get_private_predicates(graph):
    predicates = set()
    for s, p, o in graph:
        if str(p).startswith(str(EX)):
            predicates.add(p)
    return sorted(predicates, key=lambda x: uri_local_name(x))

def get_sameas_map(graph):
    """
    Retourne un dict:
    private_entity_uri -> wikidata_entity_uri
    """
    sameas_map = {}
    for s, p, o in graph.triples((None, OWL.sameAs, None)):
        if str(s).startswith(str(EX)) and str(o).startswith(WD):
            sameas_map[s] = o
    return sameas_map

def find_predicate_by_label(keyword, limit=20):
    """
    Cherche des propriétés Wikidata dont le label contient le mot-clé.
    """
    query = f"""
    SELECT ?property ?propertyLabel WHERE {{
      ?property a wikibase:Property .
      ?property rdfs:label ?propertyLabel .
      FILTER(LANG(?propertyLabel) = "en") .
      FILTER(CONTAINS(LCASE(?propertyLabel), "{keyword.lower()}"))
    }}
    LIMIT {limit}
    """
    sparql.setQuery(query)
    results = sparql.query().convert()

    candidates = []
    for r in results["results"]["bindings"]:
        candidates.append({
            "property_uri": r["property"]["value"],
            "property_label": r["propertyLabel"]["value"]
        })
    return candidates

def find_predicates_between_aligned_entities(wd_subject, wd_object, limit=20):
    """
    Cherche les propriétés existantes entre wd_subject et wd_object.
    Conforme à la logique de la consigne.
    """
    query = f"""
    SELECT ?property ?propertyLabel WHERE {{
      {wd_subject.n3()} ?property {wd_object.n3()} .
      ?property a wikibase:Property .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    LIMIT {limit}
    """
    sparql.setQuery(query)
    results = sparql.query().convert()

    candidates = []
    for r in results["results"]["bindings"]:
        candidates.append({
            "property_uri": r["property"]["value"],
            "property_label": r.get("propertyLabel", {}).get("value", "")
        })
    return candidates

def get_aligned_triples_for_predicate(graph, predicate, sameas_map):
    """
    Retourne les triples (s, p, o) pour lesquels s et o ont tous les deux un owl:sameAs vers Wikidata.
    """
    aligned = []
    for s, p, o in graph.triples((None, predicate, None)):
        if s in sameas_map and o in sameas_map:
            aligned.append((s, p, o, sameas_map[s], sameas_map[o]))
    return aligned

# =========================
# MANUAL SEED KEYWORDS
# =========================
# Ici tu peux associer chaque prédicat privé à un mot-clé de recherche.
predicate_keywords = {
    "directedBy": "director",
    "actedIn": "cast",
    "hasGenre": "genre",
    "wonAward": "award",
    "producedIn": "country",
}

# =========================
# OPTIONAL MANUAL VALIDATION TARGETS
# =========================
# Ici tu peux fixer manuellement la meilleure propriété si tu la connais.
manual_validation = {
    "wonAward": "http://www.wikidata.org/prop/direct/P166",   # award received
    "directedBy": "http://www.wikidata.org/prop/direct/P57",  # director
    "hasGenre": "http://www.wikidata.org/prop/direct/P136",   # genre
    "producedIn": "http://www.wikidata.org/prop/direct/P495", # country of origin
    # actedIn n'a pas toujours un équivalent direct parfait en WDT dans ce sens
}

# =========================
# MAIN
# =========================
private_predicates = get_private_predicates(g)
sameas_map = get_sameas_map(g)

print(f"Found {len(private_predicates)} private predicates")
print(f"Found {len(sameas_map)} aligned entities with owl:sameAs\n")

rows = []

for pred in private_predicates:
    pred_name = uri_local_name(pred)
    print("=" * 80)
    print(f"PRIVATE PREDICATE: {pred_name}")

    # 1) Recherche sémantique par label
    keyword = predicate_keywords.get(pred_name, pred_name)
    label_candidates = find_predicate_by_label(keyword, limit=10)

    print(f"\nCandidates by label for keyword '{keyword}':")
    for c in label_candidates[:5]:
        print(f"  - {c['property_uri']} -> {c['property_label']}")

    # 2) Recherche via triples alignés s --p--> o
    aligned_triples = get_aligned_triples_for_predicate(g, pred, sameas_map)

    relation_candidates = []
    if aligned_triples:
        s, p, o, wd_s, wd_o = aligned_triples[0]
        relation_candidates = find_predicates_between_aligned_entities(wd_s, wd_o, limit=10)

        print(f"\nCandidates between aligned entities:")
        print(f"  Subject: {uri_local_name(s)} -> {wd_s}")
        print(f"  Object : {uri_local_name(o)} -> {wd_o}")
        for c in relation_candidates[:5]:
            print(f"  - {c['property_uri']} -> {c['property_label']}")
    else:
        print("\nNo aligned (subject, object) pair found for this predicate.")

    # 3) Choix final (manuel ou laissé vide)
    selected_uri = manual_validation.get(pred_name, "")
    selected_type = ""

    if selected_uri:
        if pred_name in ["wonAward", "directedBy", "hasGenre", "producedIn"]:
            selected_type = "owl:equivalentProperty"
        else:
            selected_type = "manual_review"

    rows.append({
        "Private Predicate": str(pred),
        "Predicate Name": pred_name,
        "Keyword Used": keyword,
        "Top Label Candidate URI": label_candidates[0]["property_uri"] if label_candidates else "",
        "Top Label Candidate": label_candidates[0]["property_label"] if label_candidates else "",
        "Top Relation Candidate URI": relation_candidates[0]["property_uri"] if relation_candidates else "",
        "Top Relation Candidate": relation_candidates[0]["property_label"] if relation_candidates else "",
        "Selected Alignment URI": selected_uri,
        "Alignment Type": selected_type
    })

# =========================
# SAVE CSV
# =========================
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_FILE, index=False)

print("\nDone.")
print(f"Predicate alignment table saved as: {OUTPUT_FILE}")

Found 5 private predicates
Found 34 aligned entities with owl:sameAs

PRIVATE PREDICATE: actedIn

Candidates by label for keyword 'cast':
  - http://www.wikidata.org/entity/P449 -> original broadcaster
  - http://www.wikidata.org/entity/P161 -> cast member
  - http://www.wikidata.org/entity/P3301 -> broadcast by
  - http://www.wikidata.org/entity/P4818 -> Panoptikum podcast ID
  - http://www.wikidata.org/entity/P5840 -> NPR podcast ID

Candidates between aligned entities:
  Subject: LeonardoDiCaprio -> http://www.wikidata.org/entity/Q38111
  Object : Inception -> http://www.wikidata.org/entity/Q43361
PRIVATE PREDICATE: directedBy

Candidates by label for keyword 'director':
  - http://www.wikidata.org/entity/P633 -> Quebec cultural heritage directory ID
  - http://www.wikidata.org/entity/P57 -> director
  - http://www.wikidata.org/entity/P344 -> director of photography
  - http://www.wikidata.org/entity/P4227 -> Indonesian Small Islands Directory ID
  - http://www.wikidata.org/entity/P

# Step 4 - KB Expansion via SPARQL


In [1]:
from SPARQLWrapper import SPARQLWrapper, JSON
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL
from collections import Counter, defaultdict, deque
import pandas as pd
import time

# =========================
# CONFIG
# =========================
INPUT_FILE = r"C:\ESILV\Web\TD4\private_kb_aligned.ttl"
OUTPUT_FILE = "expanded_kb.ttl"
STATS_FILE = "expanded_kb_stats.txt"

EX = Namespace("http://example.org/")
WD = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"

# objectifs calibrés pour respecter la consigne
MAX_CORE_ENTITIES = 200
ONE_HOP_LIMIT = 4000
PREDICATE_CONTROLLED_LIMIT = 120000
TWO_HOP_LIMIT = 80000

SLEEP_BETWEEN_QUERIES = 1.0

# filtrage final beaucoup moins destructeur
TARGET_RELATION_COUNT = 100
MIN_ENTITY_FREQUENCY = 2
MAX_LITERAL_LENGTH = 120
MAX_FIRST_HOP_NEIGHBORS_FOR_2HOP = 1500

# prédicats WDT autorisés : liste élargie pour atteindre 50+ relations
ALLOWED_WDT_PROPERTIES = {
    "P31", "P279", "P361", "P527", "P1269", "P1552", "P462", "P186", "P366",
    "P1535", "P2283", "P2578", "P1056", "P1344", "P710", "P3095", "P101",
    "P2579", "P425", "P50", "P61", "P170", "P84", "P57", "P86", "P175",
    "P264", "P136", "P495", "P17", "P27", "P19", "P20", "P176", "P178",
    "P400", "P105", "P171", "P703", "P452", "P921", "P737", "P941", "P138",
    "P460", "P1889", "P641", "P106", "P39", "P69", "P108", "P463", "P102",
    "P1303", "P1412", "P1416", "P2176", "P780", "P2175", "P355", "P749",
    "P1830", "P26", "P40", "P22", "P25", "P161", "P166", "P364", "P750",
    "P272", "P58", "P915", "P840", "P179", "P155", "P156", "P577"
}

# relations privées déjà alignées
PRIVATE_TO_WDT = {
    EX.directedBy: WDT.P57,
    EX.hasGenre: WDT.P136,
    EX.wonAward: WDT.P166,
    EX.releasedIn: WDT.P577,
}

# =========================
# LOAD GRAPH
# =========================
g = Graph()
g.parse(INPUT_FILE, format="turtle")

expanded = Graph()
for triple in g:
    expanded.add(triple)

expanded.bind("ex", EX)
expanded.bind("wd", WD)
expanded.bind("wdt", WDT)
expanded.bind("owl", OWL)
expanded.bind("rdf", RDF)
expanded.bind("rdfs", RDFS)

# =========================
# SPARQL SETUP
# =========================
sparql = SPARQLWrapper(SPARQL_ENDPOINT)
sparql.setReturnFormat(JSON)
sparql.addCustomHttpHeader(
    "User-Agent",
    "ESILV-KB-Expansion/1.0 (student project)"
)

# =========================
# HELPERS
# =========================
def local_name(uri):
    uri = str(uri)
    if "#" in uri:
        return uri.split("#")[-1]
    return uri.rstrip("/").split("/")[-1]

def is_wikidata_entity(uri):
    return str(uri).startswith("http://www.wikidata.org/entity/Q")

def is_wikidata_property(uri):
    return str(uri).startswith("http://www.wikidata.org/prop/direct/P")

def run_query(query):
    sparql.setQuery(query)
    return sparql.query().convert()

def get_aligned_core_entities(graph):
    """
    Récupère les entités privées ayant un owl:sameAs vers Wikidata.
    """
    aligned = []
    for s, p, o in graph.triples((None, OWL.sameAs, None)):
        if str(s).startswith(str(EX)) and is_wikidata_entity(o):
            aligned.append((s, o))
    return aligned[:MAX_CORE_ENTITIES]

def safe_add_triple(graph, s, p, o):
    """
    Garde surtout URI->URI, et quelques petits littéraux utiles.
    """
    if isinstance(o, Literal):
        text = str(o)
        if len(text) > MAX_LITERAL_LENGTH:
            return False
        if text.strip() == "":
            return False
    graph.add((s, p, o))
    return True

def add_one_hop_for_entity(graph, wd_entity, limit=2000):
    """
    1-hop expansion depuis une entité ancrée.
    """
    query = f"""
    SELECT ?p ?o WHERE {{
      <{wd_entity}> ?p ?o .
      FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
    }}
    LIMIT {limit}
    """
    results = run_query(query)

    added = 0
    for r in results["results"]["bindings"]:
        p = r["p"]["value"]
        o = r["o"]["value"]
        o_type = r["o"]["type"]

        p_local = local_name(p)
        if p_local not in ALLOWED_WDT_PROPERTIES:
            continue

        subj = URIRef(wd_entity)
        pred = URIRef(p)

        if o_type == "uri":
            obj = URIRef(o)
        else:
            obj = Literal(o)

        if safe_add_triple(graph, subj, pred, obj):
            added += 1

    return added

def add_predicate_controlled_expansion(graph, property_id, limit=5000):
    """
    Expansion contrôlée sur un prédicat Wikidata spécifique.
    """
    query = f"""
    SELECT ?s ?o WHERE {{
      ?s wdt:{property_id} ?o .
      FILTER(STRSTARTS(STR(?s), "http://www.wikidata.org/entity/Q"))
      FILTER(STRSTARTS(STR(?o), "http://www.wikidata.org/entity/Q"))
    }}
    LIMIT {limit}
    """
    results = run_query(query)

    added = 0
    pred = URIRef(f"http://www.wikidata.org/prop/direct/{property_id}")

    for r in results["results"]["bindings"]:
        s = r["s"]["value"]
        o = r["o"]["value"]

        if safe_add_triple(graph, URIRef(s), pred, URIRef(o)):
            added += 1

    return added

def add_two_hop_from_core_neighbors(graph, core_entities, limit=80000):
    """
    2-hop plus large :
    on part des voisins URI déjà reliés aux core entities,
    puis on récupère des propriétés autorisées de ces voisins.
    """
    core_entity_uris = {str(wd_entity) for _, wd_entity in core_entities}

    first_hop_neighbors = set()
    for s, p, o in graph:
        if isinstance(s, URIRef) and isinstance(o, URIRef):
            if str(s) in core_entity_uris and is_wikidata_entity(o):
                first_hop_neighbors.add(str(o))

    neighbors = list(first_hop_neighbors)
    if not neighbors:
        return 0

    neighbors = neighbors[:MAX_FIRST_HOP_NEIGHBORS_FOR_2HOP]

    total_added = 0
    chunk_size = 100

    for i in range(0, len(neighbors), chunk_size):
        chunk = neighbors[i:i + chunk_size]
        values_block = " ".join(f"<{n}>" for n in chunk)

        query = f"""
        SELECT ?s ?p ?o WHERE {{
          VALUES ?s {{ {values_block} }}
          ?s ?p ?o .
          FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
        }}
        LIMIT {limit // max(1, (len(neighbors) // chunk_size + 1))}
        """

        try:
            results = run_query(query)
            for r in results["results"]["bindings"]:
                s = r["s"]["value"]
                p = r["p"]["value"]
                o = r["o"]["value"]
                o_type = r["o"]["type"]

                p_local = local_name(p)
                if p_local not in ALLOWED_WDT_PROPERTIES:
                    continue

                s_ref = URIRef(s)
                p_ref = URIRef(p)

                if o_type == "uri":
                    o_ref = URIRef(o)
                else:
                    o_ref = Literal(o)

                if safe_add_triple(graph, s_ref, p_ref, o_ref):
                    total_added += 1

        except Exception as e:
            print(f"  ERROR in 2-hop chunk {i//chunk_size}: {e}")

        time.sleep(SLEEP_BETWEEN_QUERIES)

    return total_added

def add_neighbor_expansion(graph, core_entities, per_neighbor_limit=300, max_neighbors=1000):
    """
    Expansion supplémentaire :
    on reprend les voisins Wikidata déjà trouvés autour des entités coeur
    et on les développe chacun localement.
    """
    core_entity_uris = {str(wd_entity) for _, wd_entity in core_entities}

    neighbors = set()
    for s, p, o in graph:
        if isinstance(s, URIRef) and isinstance(o, URIRef):
            if str(s) in core_entity_uris and is_wikidata_entity(o):
                neighbors.add(str(o))

    neighbors = list(neighbors)[:max_neighbors]

    total_added = 0
    for idx, neighbor in enumerate(neighbors, 1):
        query = f"""
        SELECT ?p ?o WHERE {{
          <{neighbor}> ?p ?o .
          FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
        }}
        LIMIT {per_neighbor_limit}
        """
        try:
            results = run_query(query)
            for r in results["results"]["bindings"]:
                p = r["p"]["value"]
                o = r["o"]["value"]
                o_type = r["o"]["type"]

                p_local = local_name(p)
                if p_local not in ALLOWED_WDT_PROPERTIES:
                    continue

                subj = URIRef(neighbor)
                pred = URIRef(p)

                if o_type == "uri":
                    obj = URIRef(o)
                else:
                    obj = Literal(o)

                if safe_add_triple(graph, subj, pred, obj):
                    total_added += 1

        except Exception as e:
            print(f"  ERROR in neighbor expansion {idx}: {e}")

        time.sleep(SLEEP_BETWEEN_QUERIES)

    return total_added

def remove_literal_heavy_triples(graph):
    to_remove = []
    for s, p, o in graph:
        if isinstance(o, Literal):
            txt = str(o)
            if len(txt) > MAX_LITERAL_LENGTH:
                to_remove.append((s, p, o))
    for triple in to_remove:
        graph.remove(triple)
    return len(to_remove)

def remove_non_wanted_properties(graph):
    """
    On garde :
    - propriétés privées
    - owl/rdf/rdfs utiles
    - propriétés WDT autorisées
    """
    to_remove = []
    for s, p, o in graph:
        p_str = str(p)

        keep = False
        if p_str.startswith(str(EX)):
            keep = True
        elif p in {
            RDF.type, RDFS.label, OWL.sameAs, OWL.equivalentProperty,
            RDFS.subPropertyOf, RDFS.subClassOf
        }:
            keep = True
        elif p_str.startswith("http://www.wikidata.org/prop/direct/"):
            if local_name(p) in ALLOWED_WDT_PROPERTIES:
                keep = True

        if not keep:
            to_remove.append((s, p, o))

    for triple in to_remove:
        graph.remove(triple)
    return len(to_remove)

def filter_by_global_frequency(graph, target_relation_count=100, min_entity_frequency=2):
    """
    Filtrage modéré :
    - garde les top relations WDT
    - ne supprime pas agressivement selon la fréquence des entités
    """
    predicate_freq = Counter()

    for s, p, o in graph:
        if str(p).startswith("http://www.wikidata.org/prop/direct/"):
            predicate_freq[local_name(p)] += 1

    top_predicates = {
        pred for pred, _ in predicate_freq.most_common(target_relation_count)
    }

    to_remove = []
    for s, p, o in graph:
        p_str = str(p)

        if p_str.startswith("http://www.wikidata.org/prop/direct/"):
            if local_name(p) not in top_predicates:
                to_remove.append((s, p, o))

    for triple in to_remove:
        graph.remove(triple)

    return len(to_remove), top_predicates

def prune_low_degree_entities(graph, min_degree=2):
    """
    Supprime seulement les triplets Wikidata où
    sujet ET objet URI sont tous les deux trop rares.
    Version douce pour ne pas casser le volume.
    """
    entity_freq = Counter()

    for s, p, o in graph:
        if isinstance(s, URIRef):
            entity_freq[str(s)] += 1
        if isinstance(o, URIRef):
            entity_freq[str(o)] += 1

    to_remove = []
    for s, p, o in graph:
        p_str = str(p)

        # on ne touche pas au graphe privé / schéma
        if p_str.startswith(str(EX)):
            continue
        if p in {
            RDF.type, RDFS.label, OWL.sameAs, OWL.equivalentProperty,
            RDFS.subPropertyOf, RDFS.subClassOf
        }:
            continue

        # on prune seulement les triples WDT très faibles
        if p_str.startswith("http://www.wikidata.org/prop/direct/"):
            s_low = isinstance(s, URIRef) and entity_freq[str(s)] < min_degree
            o_low = isinstance(o, URIRef) and entity_freq[str(o)] < min_degree

            # on enlève seulement si les deux côtés sont faibles
            if s_low and o_low:
                to_remove.append((s, p, o))

    for triple in to_remove:
        graph.remove(triple)

    return len(to_remove)

def prune_leaf_objects(graph, protected_entities=None, passes=3):
    """
    Supprime les objets URI feuilles :
    - URI apparaissant une seule fois
    - uniquement sur les triplets WDT
    - sans toucher aux entités protégées (core entities)
    On peut faire plusieurs passes légères.
    """
    if protected_entities is None:
        protected_entities = set()

    total_removed = 0

    for _ in range(passes):
        entity_freq = Counter()

        for s, p, o in graph:
            if isinstance(s, URIRef):
                entity_freq[str(s)] += 1
            if isinstance(o, URIRef):
                entity_freq[str(o)] += 1

        to_remove = []
        for s, p, o in graph:
            p_str = str(p)

            if not p_str.startswith("http://www.wikidata.org/prop/direct/"):
                continue

            if not isinstance(o, URIRef):
                continue

            o_str = str(o)

            # ne pas toucher au noyau aligné
            if o_str in protected_entities:
                continue
            if isinstance(s, URIRef) and str(s) in protected_entities:
                continue

            # on coupe seulement les objets-feuilles
            if entity_freq[o_str] == 1:
                to_remove.append((s, p, o))

        if not to_remove:
            break

        for triple in to_remove:
            graph.remove(triple)

        total_removed += len(to_remove)

    return total_removed

aligned_core = get_aligned_core_entities(g)
print(f"Aligned core entities selected: {len(aligned_core)}")

protected_entities = {str(wd_entity) for _, wd_entity in aligned_core}

def keep_largest_connected_component(graph):
    """
    Approximation simple de connectivité sur les triplets URI-URI.
    """
    adjacency = defaultdict(set)
    nodes = set()

    for s, p, o in graph:
        if isinstance(s, URIRef) and isinstance(o, URIRef):
            adjacency[s].add(o)
            adjacency[o].add(s)
            nodes.add(s)
            nodes.add(o)

    if not nodes:
        return 0

    visited = set()
    largest_component = set()

    for node in nodes:
        if node in visited:
            continue

        comp = set()
        queue = deque([node])
        visited.add(node)

        while queue:
            cur = queue.popleft()
            comp.add(cur)
            for neigh in adjacency[cur]:
                if neigh not in visited:
                    visited.add(neigh)
                    queue.append(neigh)

        if len(comp) > len(largest_component):
            largest_component = comp

    to_remove = []
    for s, p, o in graph:
        s_ok = not isinstance(s, URIRef) or s in largest_component
        o_ok = not isinstance(o, URIRef) or o in largest_component
        if not (s_ok and o_ok):
            to_remove.append((s, p, o))

    for triple in to_remove:
        graph.remove(triple)

    return len(to_remove)

def stats(graph):
    triples = len(graph)

    entities = set()
    relations = set()

    for s, p, o in graph:
        if isinstance(s, URIRef):
            entities.add(s)
        if isinstance(o, URIRef):
            entities.add(o)
        if isinstance(p, URIRef):
            relations.add(p)

    return triples, len(entities), len(relations)

# =========================
# STEP 4A — ANCHORED 1-HOP EXPANSION
# =========================
aligned_core = get_aligned_core_entities(g)
print(f"Aligned core entities selected: {len(aligned_core)}")

total_added_1hop = 0
for private_entity, wd_entity in aligned_core:
    print(f"1-hop expanding from {private_entity} -> {wd_entity}")
    try:
        added = add_one_hop_for_entity(expanded, str(wd_entity), limit=ONE_HOP_LIMIT)
        total_added_1hop += added
        print(f"  added {added} triples")
    except Exception as e:
        print(f"  ERROR: {e}")
    time.sleep(SLEEP_BETWEEN_QUERIES)

print(f"\nTotal 1-hop triples added: {total_added_1hop}")

# =========================
# STEP 4B — PREDICATE-CONTROLLED EXPANSION
# =========================
predicate_targets = sorted(ALLOWED_WDT_PROPERTIES)

total_added_pred = 0
per_property_limit = max(300, PREDICATE_CONTROLLED_LIMIT // len(predicate_targets))

for pid in predicate_targets:
    print(f"Predicate-controlled expansion with {pid}")
    try:
        added = add_predicate_controlled_expansion(
            expanded,
            pid,
            limit=per_property_limit
        )
        total_added_pred += added
        print(f"  added {added} triples")
    except Exception as e:
        print(f"  ERROR: {e}")
    time.sleep(SLEEP_BETWEEN_QUERIES)

print(f"\nTotal predicate-controlled triples added: {total_added_pred}")

# =========================
# STEP 4C — GENERALIZED 2-HOP EXPANSION
# =========================
print("2-hop expansion from first-hop Wikidata neighbors")
try:
    total_added_2hop = add_two_hop_from_core_neighbors(
        expanded,
        aligned_core,
        limit=TWO_HOP_LIMIT
    )
except Exception as e:
    total_added_2hop = 0
    print("  ERROR:", e)

print(f"Total 2-hop triples added: {total_added_2hop}")

# =========================
# STEP 4D — CLEANING
# =========================

# =========================
# STEP 4D — NEIGHBOR EXPANSION
# =========================
print("Neighbor expansion from first-hop neighbors")
try:
    total_added_neighbors = add_neighbor_expansion(
        expanded,
        aligned_core,
        per_neighbor_limit=250,
        max_neighbors=1000
    )
except Exception as e:
    total_added_neighbors = 0
    print("  ERROR:", e)

print(f"Total neighbor-expansion triples added: {total_added_neighbors}")

print("\nCleaning graph...")

removed_props = remove_non_wanted_properties(expanded)
print(f"Removed non-wanted property triples: {removed_props}")

removed_literals = remove_literal_heavy_triples(expanded)
print(f"Removed literal-heavy triples: {removed_literals}")

removed_freq, kept_top_predicates = filter_by_global_frequency(
    expanded,
    target_relation_count=TARGET_RELATION_COUNT,
    min_entity_frequency=MIN_ENTITY_FREQUENCY
)
print(f"Removed low-frequency / non-top-predicate triples: {removed_freq}")
print(f"Top WDT predicates kept: {len(kept_top_predicates)}")

removed_leaf_objects = prune_leaf_objects(
    expanded,
    protected_entities=protected_entities,
    passes=3
)
print(f"Removed leaf-object triples: {removed_leaf_objects}")

removed_disconnected = 0
print("Removed disconnected triples: 0 (disabled to preserve graph volume)")

print("Duplicate triples automatically handled by rdflib.")

# =========================
# STEP 4E — STATS
# =========================
triple_count, entity_count, relation_count = stats(expanded)

stats_text = f"""Expanded KB statistics
======================
Triplets: {triple_count}
Entities: {entity_count}
Relations: {relation_count}

Target requirements:
- Triplets: 50,000 -- 200,000
- Entities: 5,000 -- 30,000
- Relations: 50 -- 200
"""

with open(STATS_FILE, "w", encoding="utf-8") as f:
    f.write(stats_text)

expanded.serialize(OUTPUT_FILE, format="turtle")

print("\nDone.")
print(stats_text)
print(f"Expanded KB saved as: {OUTPUT_FILE}")
print(f"Statistics saved as: {STATS_FILE}")

Aligned core entities selected: 38
Aligned core entities selected: 38
1-hop expanding from http://example.org/Avatar -> http://www.wikidata.org/entity/Q48460
  added 13 triples
1-hop expanding from http://example.org/BradPitt -> http://www.wikidata.org/entity/Q35332
  added 43 triples
1-hop expanding from http://example.org/CillianMurphy -> http://www.wikidata.org/entity/Q313949
  added 15 triples
1-hop expanding from http://example.org/Crime -> http://www.wikidata.org/entity/Q959790
  added 3 triples
1-hop expanding from http://example.org/DenisVilleneuve -> http://www.wikidata.org/entity/Q3772
  added 62 triples
1-hop expanding from http://example.org/Dune -> http://www.wikidata.org/entity/Q20050538
  added 2 triples
1-hop expanding from http://example.org/FightClub -> http://www.wikidata.org/entity/Q190908
  added 59 triples
1-hop expanding from http://example.org/Gladiator -> http://www.wikidata.org/entity/Q128518
  added 78 triples
1-hop expanding from http://example.org/GoldenGlo

In [2]:
from rdflib import Graph, Namespace, URIRef
from rdflib.namespace import RDF, RDFS, OWL
from collections import Counter

# =========================
# FILES
# =========================
INPUT_FILE = "expanded_kb.ttl"          # ton graphe déjà généré
ORIGINAL_FILE = r"C:\ESILV\Web\TD4\private_kb_aligned.ttl"   # graphe initial
OUTPUT_FILE = "expanded_kb_pruned.ttl"
STATS_FILE = "expanded_kb_pruned_stats.txt"

EX = Namespace("http://example.org/")
WD = Namespace("http://www.wikidata.org/entity/")
WDT_PREFIX = "http://www.wikidata.org/prop/direct/"

TARGET_MIN_TRIPLES = 50000
TARGET_MAX_ENTITIES = 30000

# =========================
# HELPERS
# =========================
def is_uri(x):
    return isinstance(x, URIRef)

def is_wikidata_entity(x):
    return is_uri(x) and str(x).startswith("http://www.wikidata.org/entity/Q")

def is_wdt_property(x):
    return str(x).startswith(WDT_PREFIX)

def stats(graph):
    entities = set()
    relations = set()
    for s, p, o in graph:
        if is_uri(s):
            entities.add(s)
        if is_uri(o):
            entities.add(o)
        if is_uri(p):
            relations.add(p)
    return len(graph), len(entities), len(relations)

# =========================
# LOAD
# =========================
g = Graph()
g.parse(INPUT_FILE, format="turtle")

g0 = Graph()
g0.parse(ORIGINAL_FILE, format="turtle")

print("Initial stats:", stats(g))

# =========================
# PROTECTED ENTITIES
# =========================
protected = set()

# protéger toutes les entités du graphe original
for s, p, o in g0:
    if is_uri(s):
        protected.add(str(s))
    if is_uri(o):
        protected.add(str(o))

# protéger aussi les entités wikidata alignées
for s, p, o in g0.triples((None, OWL.sameAs, None)):
    if is_wikidata_entity(o):
        protected.add(str(o))

print("Protected entities:", len(protected))

# =========================
# ADAPTIVE LOCAL PRUNING
# =========================
# idée :
# 1) enlever les triples WDT dont l'objet URI est une feuille rare non protégée
# 2) si besoin, enlever les triples WDT dont sujet+objet sont tous les deux faibles
# 3) on s'arrête dès qu'on rentre dans la cible ou qu'on risque de tomber sous 50k triples

threshold_schedule = [2, 3, 4, 5, 6, 8, 10, 12]
pass_num = 0

for k in threshold_schedule:
    pass_num += 1

    # recalcul des degrés URI
    degree = Counter()
    for s, p, o in g:
        if is_uri(s):
            degree[str(s)] += 1
        if is_uri(o):
            degree[str(o)] += 1

    current_triples, current_entities, current_relations = stats(g)
    print(f"\nPass {pass_num} | threshold={k} | before={current_triples, current_entities, current_relations}")

    to_remove_stage1 = []
    to_remove_stage2 = []

    for s, p, o in g:
        # on ne prune que les triples Wikidata directs
        if not is_wdt_property(p):
            continue

        # stage 1 : enlever les objets feuilles non protégés
        if is_wikidata_entity(o):
            o_str = str(o)
            if o_str not in protected and degree[o_str] < k:
                to_remove_stage1.append((s, p, o))
                continue

        # stage 2 : si sujet et objet sont tous les deux faibles et non protégés
        s_low = is_wikidata_entity(s) and str(s) not in protected and degree[str(s)] < k
        o_low = is_wikidata_entity(o) and str(o) not in protected and degree[str(o)] < k

        if s_low and o_low:
            to_remove_stage2.append((s, p, o))

    # appliquer stage 1
    projected_triples = len(g) - len(to_remove_stage1)
    if projected_triples >= TARGET_MIN_TRIPLES:
        for t in to_remove_stage1:
            g.remove(t)

    # recalcul rapide si besoin
    if stats(g)[1] > TARGET_MAX_ENTITIES:
        projected_triples = len(g) - len(to_remove_stage2)
        if projected_triples >= TARGET_MIN_TRIPLES:
            for t in to_remove_stage2:
                g.remove(t)

    current_triples, current_entities, current_relations = stats(g)
    print(f"Removed stage1={len(to_remove_stage1)}, stage2={len(to_remove_stage2)}")
    print(f"After pass {pass_num}: {current_triples, current_entities, current_relations}")

    # stop dès que la cible est atteinte
    if current_entities <= TARGET_MAX_ENTITIES and current_triples >= TARGET_MIN_TRIPLES:
        break

# =========================
# FINAL SAVE
# =========================
triple_count, entity_count, relation_count = stats(g)

stats_text = f"""Expanded KB statistics
======================
Triplets: {triple_count}
Entities: {entity_count}
Relations: {relation_count}

Target requirements:
- Triplets: 50,000 -- 200,000
- Entities: 5,000 -- 30,000
- Relations: 50 -- 200
"""

with open(STATS_FILE, "w", encoding="utf-8") as f:
    f.write(stats_text)

g.serialize(OUTPUT_FILE, format="turtle")

print("\nDone.")
print(stats_text)
print(f"Pruned KB saved as: {OUTPUT_FILE}")
print(f"Statistics saved as: {STATS_FILE}")

Initial stats: (123022, 106462, 86)
Protected entities: 85

Pass 1 | threshold=2 | before=(123022, 106462, 86)
Removed stage1=20, stage2=0
After pass 1: (123002, 106441, 86)

Pass 2 | threshold=3 | before=(123002, 106441, 86)
Removed stage1=3662, stage2=0
After pass 2: (119340, 103141, 86)

Pass 3 | threshold=4 | before=(119340, 103141, 86)
Removed stage1=2876, stage2=0
After pass 3: (116464, 100777, 86)

Pass 4 | threshold=5 | before=(116464, 100777, 86)
Removed stage1=3559, stage2=0
After pass 4: (112905, 98525, 86)

Pass 5 | threshold=6 | before=(112905, 98525, 86)
Removed stage1=2099, stage2=0
After pass 5: (110806, 97026, 86)

Pass 6 | threshold=8 | before=(110806, 97026, 86)
Removed stage1=3360, stage2=0
After pass 6: (107446, 94627, 86)

Pass 7 | threshold=10 | before=(107446, 94627, 86)
Removed stage1=2759, stage2=0
After pass 7: (104687, 92593, 86)

Pass 8 | threshold=12 | before=(104687, 92593, 86)
Removed stage1=2058, stage2=0
After pass 8: (102629, 91167, 86)

Done.
Expande

In [5]:
from rdflib import Graph, Namespace, URIRef
from rdflib.namespace import RDF, OWL, RDFS

INPUT_FILE = "expanded_kb_pruned.ttl"
ALIGNMENT_FILE = "alignment.ttl"
ONTOLOGY_FILE = "ontology.ttl"

EX = Namespace("http://example.org/")

g = Graph()
g.parse(INPUT_FILE, format="turtle")

alignment_graph = Graph()
ontology_graph = Graph()

alignment_graph.bind("ex", EX)
ontology_graph.bind("ex", EX)

for s, p, o in g:
    
    # =========================
    # ALIGNMENT
    # =========================
    if p in [OWL.sameAs, OWL.equivalentProperty]:
        alignment_graph.add((s, p, o))
    
    # =========================
    # ONTOLOGY (only private namespace)
    # =========================
    if not str(s).startswith(str(EX)):
        continue
    
    # Classes
    if p == RDF.type and o == RDFS.Class:
        ontology_graph.add((s, p, o))
    
    # Properties
    elif p == RDF.type and o in [OWL.ObjectProperty, RDF.Property]:
        ontology_graph.add((s, p, o))
    
    # Domain / Range
    elif p in [RDFS.domain, RDFS.range]:
        ontology_graph.add((s, p, o))
    
    # Hierarchy
    elif p in [RDFS.subClassOf, RDFS.subPropertyOf]:
        ontology_graph.add((s, p, o))
    
    # Labels (important)
    elif p == RDFS.label:
        ontology_graph.add((s, p, o))

# =========================
# SAVE
# =========================
alignment_graph.serialize(ALIGNMENT_FILE, format="turtle")
ontology_graph.serialize(ONTOLOGY_FILE, format="turtle")

print("Alignment file created:", ALIGNMENT_FILE)
print("Ontology file created:", ONTOLOGY_FILE)

Alignment file created: alignment.ttl
Ontology file created: ontology.ttl


# Thank you for following our study